# SLD Component Detection — D-FINE Inference Notebook

**Before running:** upload `best_stg2.pth` when stage 2 is the checkpoint you want to evaluate; the notebook falls back to `best_stg1.pth` only when stage 2 is absent.

D-FINE saves separate stage-1 and stage-2 best checkpoints; this notebook prefers `best_stg2.pth` and prints the selected file and epoch after loading.

**Two important checks before trusting results:**
- This notebook **reconstructs** the model architecture config from your training log output, since the original custom training YAML wasn't available here. If you still have that file (the one passed to `train.py -c ...`), upload it and point `CONFIG_PATH` at it in Step 3 instead — that guarantees an exact match.
- Your training log showed `remap_mscoco_category: True`. That flag is meant for real MS-COCO category IDs, not custom datasets — if detections come back with obviously wrong/scrambled class names, this is the first thing to check (would need a retrain with it set to `False`).

**Two modes:**
- Mode 1: Tiled 5×5 with overlap
- Mode 2: Tiled 4×4 with overlap (recommended for large SLD sheets)


In [ ]:
# ══════════════════════════════════════════════════════
# STEP 1 — Install D-FINE
# ══════════════════════════════════════════════════════
import os
import json

if not os.path.exists('/content/D-FINE'):
    !git clone --depth 1 https://github.com/Peterande/D-FINE.git /content/D-FINE

%cd /content/D-FINE
!pip install -q -r requirements.txt
!pip install -q supervision Pillow

import importlib
for pkg in ['torch', 'torchvision', 'supervision']:
    m = importlib.import_module(pkg)
    print(f'ok {pkg} {getattr(m, "__version__", "")}')


/content/D-FINE
ok torch 2.11.0+cu128
ok torchvision 0.26.0+cu128
ok supervision 0.29.1


In [ ]:
# ══════════════════════════════════════════════════════
# STEP 2 — Config
# ══════════════════════════════════════════════════════
import os

CHECKPOINT_CANDIDATES = ['/content/best_stg2.pth', '/content/best_stg1.pth']
DFINE_WEIGHTS_PATH = next((p for p in CHECKPOINT_CANDIDATES if os.path.exists(p)), None)
if DFINE_WEIGHTS_PATH is None:
    raise FileNotFoundError(f'Upload one of these checkpoints: {CHECKPOINT_CANDIDATES}')

NUM_CLASSES   = 30
RESOLUTION    = 640
CONFIDENCE    = 0.20
GRID_SIZE     = 4
OVERLAP       = 0.20
IOU_THRESHOLD = 0.50
REFERENCE_CLASS_PRIORITY = ['Circuit Breaker', 'Current Transformer', 'Fuse']
TARGET_REFERENCE_HEIGHT = 60.0
REFERENCE_SHEET_WIDTH = 14044
REFERENCE_MEDIAN_SYMBOL_PX = 210.0

# Fallback only. Prefer loading categories from the processed COCO JSON below.
CLASS_NAMES = [
    'ACB', 'ATS', 'Ammeter', 'UNKNOWN_ID_4', 'CIRCUIT BREAKER',
    'CURRENT TRANSFORMER', 'Circuit Breaker 2', 'Contractor 1',
    'DIGITAL METER', 'Digital Power Meter', 'EARTH LEAKAGE RELAY',
    'EMS', 'Earth Fault Relay', 'FUSE 1', 'FUSE 2',
    'HRC FUSE WITH BLOWN FUSE INDICATOR', 'ISOLATOR',
    'MAXIMUM DEMAND AMMETER', 'Over current Relay', 'PHASE INDICATOR LIGHTS',
    'Power Quality Meter', 'RCCB', 'SELECTOR SWITCH', 'SHUNT TRIP',
    'SINGLE PHASE UNFUSED TAP OFF UNIT', 'SURGE ARRESTOR 1', 'UNKNOWN_ID_27',
    'TIME DELAY RELAY', 'VOLTMETER', 'ZERO PHASE SEQUENCE CURRENT TRANSFORMER',
]

CATEGORY_SOURCE_CANDIDATES = [
    '/content/processed/train/_annotations.coco.json',
    '/content/processed_train_annotations.coco.json',
    '/content/_annotations.coco.json',
]
CATEGORY_SOURCE_PATH = next((p for p in CATEGORY_SOURCE_CANDIDATES if os.path.exists(p)), None)
if CATEGORY_SOURCE_PATH:
    with open(CATEGORY_SOURCE_PATH) as f:
        processed_coco = json.load(f)
    categories = sorted(processed_coco['categories'], key=lambda c: c['id'])
    CLASS_NAMES = [c['name'] for c in categories]
else:
    print('WARNING: processed COCO categories not found; fallback names are not authoritative.')

assert os.path.exists(DFINE_WEIGHTS_PATH), f'D-FINE weights not found: {DFINE_WEIGHTS_PATH}'
assert len(CLASS_NAMES) == NUM_CLASSES, (
    f'CLASS_NAMES has {len(CLASS_NAMES)} entries but NUM_CLASSES={NUM_CLASSES}. '
    'Upload the processed COCO JSON so labels follow the trained category order.'
)

print('=' * 60)
print(f'D-FINE  Weights : {DFINE_WEIGHTS_PATH}')
print(f'Num Classes     : {NUM_CLASSES}')
print(f'Resolution      : {RESOLUTION}')
print(f'Confidence      : {CONFIDENCE}')
print(f'Classes         : {CLASS_NAMES}')
print('=' * 60)


D-FINE  Weights : /content/best_stg1.pth
Num Classes     : 30
Resolution      : 640
Confidence      : 0.2
Classes         : ['ACB', 'ATS', 'Ammeter', 'UNKNOWN_ID_4', 'CIRCUIT BREAKER', 'CURRENT TRANSFORMER', 'Circuit Breaker 2', 'Contractor 1', 'DIGITAL METER', 'Digital Power Meter', 'EARTH LEAKAGE RELAY', 'EMS', 'Earth Fault Relay', 'FUSE 1', 'FUSE 2', 'HRC FUSE WITH BLOWN FUSE INDICATOR', 'ISOLATOR', 'MAXIMUM DEMAND AMMETER', 'Over current Relay', 'PHASE INDICATOR LIGHTS', 'Power Quality Meter', 'RCCB', 'SELECTOR SWITCH', 'SHUNT TRIP', 'SINGLE PHASE UNFUSED TAP OFF UNIT', 'SURGE ARRESTOR 1', 'UNKNOWN_ID_27', 'TIME DELAY RELAY', 'VOLTMETER', 'ZERO PHASE SEQUENCE CURRENT TRANSFORMER']


In [ ]:
# ══════════════════════════════════════════════════════
# STEP 3 — Build model from config + load checkpoint
# ══════════════════════════════════════════════════════
import sys
sys.path.append('/content/D-FINE')

import torch
import torch.nn as nn

# Reconstructed from your training log's resolved cfg (D-FINE-X / HGNetv2-B5).
# If you still have your ORIGINAL custom training yaml (the one passed to
# `train.py -c ...`), upload it instead and set CONFIG_PATH to that file —
# that guarantees an exact architecture match rather than a reconstruction.
RECONSTRUCTED_CFG = f'''
task: detection
model: DFINE
postprocessor: DFINEPostProcessor
num_classes: {NUM_CLASSES}
eval_spatial_size: [{RESOLUTION}, {RESOLUTION}]
use_focal_loss: True

DFINE:
  backbone: HGNetv2
  encoder: HybridEncoder
  decoder: DFINETransformer

HGNetv2:
  name: B5
  return_idx: [1, 2, 3]
  freeze_stem_only: True
  freeze_at: 0
  freeze_norm: True
  pretrained: False

HybridEncoder:
  in_channels: [512, 1024, 2048]
  feat_strides: [8, 16, 32]
  hidden_dim: 384
  use_encoder_idx: [2]
  num_encoder_layers: 1
  nhead: 8
  dim_feedforward: 2048
  dropout: 0.0
  enc_act: 'gelu'
  expansion: 1.0
  depth_mult: 1
  act: 'silu'

DFINETransformer:
  feat_channels: [384, 384, 384]
  feat_strides: [8, 16, 32]
  hidden_dim: 256
  num_levels: 3
  num_layers: 6
  eval_idx: -1
  num_queries: 300
  num_denoising: 100
  label_noise_ratio: 0.5
  box_noise_scale: 1.0
  layer_scale: 1
  num_points: [3, 6, 3]
  cross_attn_method: default
  query_select_method: default
  reg_max: 32
  reg_scale: 8

DFINEPostProcessor:
  num_top_queries: 300
'''

CONFIG_PATH = '/content/dfine_inference_config.yml'
with open(CONFIG_PATH, 'w') as f:
    f.write(RECONSTRUCTED_CFG)

from src.core import YAMLConfig

cfg = YAMLConfig(CONFIG_PATH, resume=DFINE_WEIGHTS_PATH)

checkpoint = torch.load(DFINE_WEIGHTS_PATH, map_location='cpu', weights_only=False)
print(f'Checkpoint metadata: last_epoch={checkpoint.get("last_epoch", "unknown")} date={checkpoint.get("date", "unknown")}')
if 'ema' in checkpoint:
    state = checkpoint['ema']['module']
elif 'model' in checkpoint:
    state = checkpoint['model']
else:
    state = checkpoint  # assume raw state_dict

cfg.model.load_state_dict(state)

class DFINEInferModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.model = cfg.model.deploy()
        self.postprocessor = cfg.postprocessor.deploy()

    def forward(self, images, orig_target_sizes):
        outputs = self.model(images)
        outputs = self.postprocessor(outputs, orig_target_sizes)
        return outputs  # (labels, boxes, scores)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = DFINEInferModel(cfg).to(device).eval()
print(f'Device: {device}')
print('D-FINE model loaded from', DFINE_WEIGHTS_PATH)


RuntimeError: PytorchStreamReader failed reading zip archive: failed finding central directory. This is an internal miniz error. If you are seeing this error, there is a high likelihood that your checkpoint file is corrupted. This can happen if the checkpoint was not saved properly, was transferred incorrectly, or the file was modified after saving.

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import numpy as np
import supervision as sv
import torchvision.transforms as T

transform = T.Compose([T.Resize((RESOLUTION, RESOLUTION)), T.ToTensor()])
box_annotator = sv.BoxAnnotator(thickness=3)
PALETTE = sv.ColorPalette.default()

@torch.no_grad()
def run_inference_dfine(pil_image):
    width, height = pil_image.size
    image_tensor = transform(pil_image.convert('RGB')).unsqueeze(0).to(device)
    original_size = torch.tensor([[width, height]], device=device)
    labels, boxes, scores = model(image_tensor, original_size)
    labels = labels[0].cpu().numpy().astype(int)
    boxes = boxes[0].cpu().numpy().astype(np.float32)
    scores = scores[0].cpu().numpy().astype(np.float32)
    keep = scores >= CONFIDENCE
    return sv.Detections(xyxy=boxes[keep], confidence=scores[keep], class_id=labels[keep])

def tile_image(image, grid_size=GRID_SIZE, overlap=OVERLAP):
    width, height = image.size
    tile_width, tile_height = width // grid_size, height // grid_size
    step_x = max(1, int(tile_width * (1 - overlap)))
    step_y = max(1, int(tile_height * (1 - overlap)))
    positions_x = [i * step_x for i in range(grid_size)]
    positions_y = [i * step_y for i in range(grid_size)]
    if grid_size > 1:
        positions_x[-1] = width - tile_width
        positions_y[-1] = height - tile_height
    return [(image.crop((x, y, min(x + tile_width, width), min(y + tile_height, height))), x, y)
            for y in positions_y for x in positions_x]

def merge_detections(tile_results, orig_width, orig_height):
    boxes, scores, classes = [], [], []
    for detections, x_offset, y_offset in tile_results:
        if len(detections) == 0:
            continue
        tile_boxes = detections.xyxy.copy()
        tile_boxes[:, [0, 2]] += x_offset
        tile_boxes[:, [1, 3]] += y_offset
        tile_boxes[:, [0, 2]] = np.clip(tile_boxes[:, [0, 2]], 0, orig_width)
        tile_boxes[:, [1, 3]] = np.clip(tile_boxes[:, [1, 3]], 0, orig_height)
        boxes.append(tile_boxes); scores.append(detections.confidence); classes.append(detections.class_id)
    if not boxes:
        return sv.Detections.empty()
    return sv.Detections(xyxy=np.vstack(boxes), confidence=np.concatenate(scores),
                         class_id=np.concatenate(classes).astype(int)).with_nms(IOU_THRESHOLD)

def draw_detections(pil_image, detections):
    if len(detections) == 0:
        return pil_image
    image = Image.fromarray(box_annotator.annotate(np.array(pil_image), detections)).convert('RGBA')
    overlay = Image.new('RGBA', image.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)
    font = ImageFont.load_default()
    for box, class_id, score in zip(detections.xyxy, detections.class_id, detections.confidence):
        name = CLASS_NAMES[int(class_id)] if int(class_id) < len(CLASS_NAMES) else f'class_{class_id}'
        text = f'{name} {score:.2f}'
        x1, y1 = box[0], box[1]
        left, top, right, bottom = draw.textbbox((0, 0), text, font=font)
        draw.rectangle([x1, max(0, y1 - bottom + top - 4), x1 + right - left + 4, y1], fill=(255, 255, 255, 180))
        draw.text((x1 + 2, max(0, y1 - bottom + top - 2)), text, fill=(20, 20, 20, 255), font=font)
    return Image.alpha_composite(image, overlay).convert('RGB')

In [ ]:
# ══════════════════════════════════════════════════════
# STEP 5 — Upload test image
# ══════════════════════════════════════════════════════
from google.colab import files
from IPython.display import display
import io

print('Upload your SLD image (JPG or PNG)...')
uploaded   = files.upload()
img_name   = list(uploaded.keys())[0]
test_image = Image.open(io.BytesIO(uploaded[img_name])).convert('RGB')
W, H       = test_image.size
print(f'Image: {img_name}  |  {W} x {H} px')
thumb = test_image.copy()
thumb.thumbnail((900, 900))
display(thumb)


In [ ]:
# ══════════════════════════════════════════════════════
# STEP 6 — MODE 1: Tiled 5x5 — D-FINE
# ══════════════════════════════════════════════════════
GRID_SIZE_5 = 5
W, H = test_image.size
tiles5 = tile_image(test_image, grid_size=GRID_SIZE_5, overlap=OVERLAP)
print(f'Total tiles: {len(tiles5)} ({GRID_SIZE_5}x{GRID_SIZE_5}, {int(OVERLAP*100)}% overlap)\n')

print('--- D-FINE 5x5 ---')
tile_results_dfine5 = []
for i, (tile, x_off, y_off) in enumerate(tiles5):
    det = run_inference_dfine(tile)
    tile_results_dfine5.append((det, x_off, y_off))
    print(f'  Tile {i+1:02d}/{len(tiles5)} at ({x_off},{y_off}) => {len(det)} detections')
det_dfine_5x5 = merge_detections(tile_results_dfine5, W, H)
print(f'After NMS: {len(det_dfine_5x5)} detections')
for i,(box,score,cls) in enumerate(zip(det_dfine_5x5.xyxy, det_dfine_5x5.confidence, det_dfine_5x5.class_id)):
    name = CLASS_NAMES[int(cls)] if int(cls) < len(CLASS_NAMES) else f'class_{cls}'
    print(f'  [{i+1}] {name}  conf={score:.3f}  box=[{int(box[0])},{int(box[1])},{int(box[2])},{int(box[3])}]')

result_dfine_5x5 = draw_detections(test_image, det_dfine_5x5)
out = result_dfine_5x5.copy(); out.thumbnail((1200,1200))
print('\nD-FINE 5x5 result:'); display(out)


In [ ]:
# ══════════════════════════════════════════════════════
# STEP 7 — MODE 2: Tiled 4x4 — D-FINE
# ══════════════════════════════════════════════════════
W, H = test_image.size
tiles4 = tile_image(test_image, grid_size=GRID_SIZE, overlap=OVERLAP)
print(f'Total tiles: {len(tiles4)} ({GRID_SIZE}x{GRID_SIZE}, {int(OVERLAP*100)}% overlap)\n')

print('--- D-FINE 4x4 ---')
tile_results_dfine4 = []
for i, (tile, x_off, y_off) in enumerate(tiles4):
    det = run_inference_dfine(tile)
    tile_results_dfine4.append((det, x_off, y_off))
    print(f'  Tile {i+1:02d}/{len(tiles4)} at ({x_off},{y_off}) => {len(det)} detections')
det_dfine_4x4 = merge_detections(tile_results_dfine4, W, H)
print(f'After NMS: {len(det_dfine_4x4)} detections')
for i,(box,score,cls) in enumerate(zip(det_dfine_4x4.xyxy, det_dfine_4x4.confidence, det_dfine_4x4.class_id)):
    name = CLASS_NAMES[int(cls)] if int(cls) < len(CLASS_NAMES) else f'class_{cls}'
    print(f'  [{i+1}] {name}  conf={score:.3f}  box=[{int(box[0])},{int(box[1])},{int(box[2])},{int(box[3])}]')

result_dfine_4x4 = draw_detections(test_image, det_dfine_4x4)
out = result_dfine_4x4.copy(); out.thumbnail((1200,1200))
print('\nD-FINE 4x4 result:'); display(out)

def estimate_median_symbol_px(image_width):
    return REFERENCE_MEDIAN_SYMBOL_PX * image_width / REFERENCE_SHEET_WIDTH

def detect_white_margins(image_array, threshold=240, blank_fraction=0.985):
    white = np.all(image_array > threshold, axis=2)
    top = next((row for row in range(image_array.shape[0]) if white[row].mean() < blank_fraction), 0)
    bottom = next((row for row in range(image_array.shape[0] - 1, -1, -1) if white[row].mean() < blank_fraction), image_array.shape[0] - 1)
    left = next((col for col in range(image_array.shape[1]) if white[:, col].mean() < blank_fraction), 0)
    right = next((col for col in range(image_array.shape[1] - 1, -1, -1) if white[:, col].mean() < blank_fraction), image_array.shape[1] - 1)
    return left, top, right + 1, bottom + 1

def run_adaptive_tiled_inference(full_image):
    original_width, original_height = full_image.size
    scale = TARGET_REFERENCE_HEIGHT / MEDIAN_SYMBOL_PX if MEDIAN_SYMBOL_PX > 0 else 1.0
    scaled = full_image.resize((round(original_width * scale), round(original_height * scale)), Image.Resampling.BILINEAR)
    left, top, right, bottom = detect_white_margins(np.array(scaled))
    left, top = max(0, left - 5), max(0, top - 5)
    right, bottom = min(scaled.width, right + 5), min(scaled.height, bottom + 5)
    cropped = scaled.crop((left, top, right, bottom))
    effective_median = MEDIAN_SYMBOL_PX * scale
    grid_x = max(1, round(48 * cropped.width / (RESOLUTION * effective_median)))
    grid_y = max(1, round(48 * cropped.height / (RESOLUTION * effective_median)))
    overlap = OVERLAP
    tile_width = cropped.width / (grid_x - (grid_x - 1) * overlap) if grid_x > 1 else cropped.width
    tile_height = cropped.height / (grid_y - (grid_y - 1) * overlap) if grid_y > 1 else cropped.height
    positions_x = [i * tile_width * (1 - overlap) for i in range(grid_x)]
    positions_y = [i * tile_height * (1 - overlap) for i in range(grid_y)]
    if grid_x > 1: positions_x[-1] = cropped.width - tile_width
    if grid_y > 1: positions_y[-1] = cropped.height - tile_height
    mapped = []
    for y, x in ((y, x) for y in positions_y for x in positions_x):
        x1, y1 = round(x), round(y)
        x2, y2 = min(cropped.width, round(x + tile_width)), min(cropped.height, round(y + tile_height))
        tile = cropped.crop((x1, y1, x2, y2))
        tile_width_actual, tile_height_actual = tile.size
        detections = run_inference_dfine(tile.resize((RESOLUTION, RESOLUTION), Image.Resampling.BILINEAR))
        if len(detections) == 0: continue
        boxes = detections.xyxy.copy().astype(float)
        boxes[:, [0, 2]] *= tile_width_actual / RESOLUTION
        boxes[:, [1, 3]] *= tile_height_actual / RESOLUTION
        boxes[:, [0, 2]] += x1 + left
        boxes[:, [1, 3]] += y1 + top
        boxes /= scale
        mapped.append(sv.Detections(xyxy=boxes, confidence=detections.confidence, class_id=detections.class_id))
    if not mapped: return sv.Detections.empty()
    return sv.Detections(xyxy=np.vstack([d.xyxy for d in mapped]), confidence=np.concatenate([d.confidence for d in mapped]), class_id=np.concatenate([d.class_id for d in mapped])).with_nms(IOU_THRESHOLD)


In [ ]:
# ══════════════════════════════════════════════════════
# STEP 8 — Save outputs
# ══════════════════════════════════════════════════════
print("--- D-FINE: Adaptive Tiling ---")

MEDIAN_SYMBOL_PX = estimate_median_symbol_px(test_image.width)
print(f"Median symbol size: {MEDIAN_SYMBOL_PX:.1f}px")

det_adaptive = run_adaptive_tiled_inference(test_image)

result_adaptive = draw_detections(test_image, det_adaptive)
preview = result_adaptive.copy()
preview.thumbnail((1200, 1200))
display(preview)

print(f"Total detections: {len(det_adaptive)}")
